
# 🕵️ Messy by Default
### A Hands-On Look at Real Data

*A workshop for designers, developers, and product owners at CLEVER°FRANKE*

---

Every dashboard, chart, or model starts long before anyone opens a design tool — it starts with raw, messy data and a series of quiet decisions: what to keep, what to drop, how to group, how to fill in the gaps. Those decisions shape the story just as much as the final visual does.

Today we'll work with a **real dataset of Amsterdam housing sales from Funda** — over 11,000 listings, complete with the kind of quirks, errors, and ambiguity you'd find in any real client dataset. No cleaned-up tutorial data. No `.fit()` and forget.

**What we'll do:**
1. 🧑‍💻 A 5-minute primer: data scientist vs. data analyst — what's the actual difference?
2. 📥 Load and inspect the raw data — what's actually in here?
3. 🧹 Clean it — and find out what "clean" even means
4. 📊 Explore correlations — what really drives housing price?
5. 🎭 Build two honest dashboards that tell two different stories
6. 🎁 *(Optional, for the curious)* a tiny price-prediction model
7. 💬 Discuss: what did we just do, and where does this happen in real projects?

> 💡 **How to use this notebook:** Every section has runnable code, but you don't need to write code to participate — read the markdown, look at the charts, and jump into the discussion. Cells marked **🎯 Optional / Bonus** are for anyone who wants to go further; feel free to skip them and stay in the conversation instead.



## 0. Before we start: who does what? 🧑‍🔬

You'll hear a few job titles and terms thrown around in data work — let's ground them in people you actually know.

| Role | What they actually do | At CLEVER°FRANKE |
|---|---|---|
| **Data Analyst** | Explores existing data to answer specific questions: *what happened, and why?* Mostly querying, cleaning, visualizing, and reporting on data that already exists. | This is a lot of what **Laith** does — turning raw or messy data into an answer someone can act on. |
| **Data Scientist** | Goes a step further: builds models to predict or classify *what will happen*, and designs the experiments/pipelines to test those models. More statistics, more engineering. | This is closer to **Anni's** work — building the models and pipelines that power predictions, not just descriptions. |

**A few terms you'll see today:**

| Term | Plain-language meaning |
|---|---|
| **Feature** | A column of data used to explain or predict something else (e.g. `area`, `rooms` are features for predicting `price`) |
| **Target / label** | The thing you're trying to predict (here: `price`) |
| **Correlation** | How strongly two variables move together — *not* the same as one causing the other |
| **Outlier** | A data point that's unusually far from the rest — sometimes a real rare case, sometimes a mistake |
| **Ground truth** | The "actual" correct value you're comparing predictions or cleaning decisions against — often more assumed than truly known |
| **Pipeline** | The sequence of steps data passes through: ingest → clean → transform → analyze/model → visualize |

Keep this table in mind — today's exercise sits mostly in **Data Analyst** territory (steps 1–5), with a small **Data Scientist**-style taste at the end (step 6).



## 1. Setup 📦

Run the cell below to load the libraries we'll use. Nothing exotic — just `pandas` for data handling and `matplotlib`/`seaborn` for charts.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make charts a bit bigger and cleaner by default
plt.rcParams["figure.figsize"] = (8, 5)
sns.set_style("whitegrid")

pd.set_option("display.max_columns", None)
print("Libraries loaded ✅")



## 2. Ingest: what are we even looking at? 📥

Let's load the data and take a first look — before we clean anything, before we assume anything.

**Discussion prompt (30 seconds before you run the cell):** what do you expect a housing dataset to contain? What could already be wrong with it, just from how it was collected (scraped from a website, over several years, across many different sellers)?


In [ ]:

df = pd.read_csv("funda.csv")

print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
df.head()



**A first pass with `.info()` and `.describe()`** — this is usually the very first thing a data analyst does with any new dataset. It won't tell you if the data is *right*, only what *shape* it's in.


In [ ]:

df.info()


In [ ]:

df.describe()



### 🔍 Look closely at that `.describe()` table. Anything look off?

A few things worth noticing already, just from summary statistics:
- `price` ranges from **€1,000** to **€999,999** — is a €1,000 apartment in Amsterdam plausible?
- `year_built` has a **minimum around 1005** — Amsterdam wasn't founded until roughly 1275.
- `area` has a max of over 800 m² — rare, but possible for the right property.

None of these are proven mistakes yet. That's the point: **`.describe()` gives you suspects, not verdicts.** Let's investigate.



## 3. The detective work: exploring before cleaning 🕵️

### 3.1 Missing values

Let's start with the easy check.


In [ ]:

df.isna().sum()



Good news: no missing values here. But **"no nulls" does not mean "no problems"** — a wrong number is often worse than a missing one, because it doesn't announce itself. Let's keep going.

### 3.2 The suspiciously cheap listings 💸

Let's look at the cheapest sales in the dataset.


In [ ]:

cheapest = df.nsmallest(10, "price")[["address", "area", "bedrooms", "price", "property_type", "year_built"]]
cheapest



**Discussion:** A 222 m² apartment for €1,000? A 277 m² place on Sophialaan (one of Amsterdam's most expensive streets) for €1,000?

These are almost certainly **not real market prices** — likely family transfers, data entry placeholders, or scraping errors where the real price wasn't captured. If we don't catch this, every average, chart, and prediction downstream will be quietly wrong.

Let's visualize the price distribution to see how big this problem is.


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(df["price"], bins=60, ax=axes[0])
axes[0].set_title("Price distribution (raw)")
axes[0].set_xlabel("Price (€)")

sns.histplot(df["price"], bins=60, ax=axes[1], log_scale=(False, True))
axes[1].set_title("Same data, log y-axis\n(reveals the low-price spike)")
axes[1].set_xlabel("Price (€)")

plt.tight_layout()
plt.show()



> 🎨 **Design note:** the chart on the right shows the *exact same data* as the one on the left — the only change is a log scale on the y-axis. Notice how much easier the low-price cluster is to spot. This is your first taste of today's later exercise: **the same honest data can look completely different depending on how you choose to show it.**

### 3.3 The impossible construction years 🏚️

Now the `year_built` issue.


In [ ]:

oldest = df.nsmallest(10, "year_built")[["address", "year_built", "price", "area"]]
oldest



Amsterdam does have real 17th-century canal houses (`Herengracht`, `Warmoesstraat` — those ~1600s values are plausible!). But **1005** and **1076** are not — those are almost certainly typos, likely `1905` and `1976` with a digit dropped or swapped.

**This is an important distinction for any data cleaning decision:** not every outlier is an error, and not every error looks extreme. A silent typo (`1905` → `1005`) is a real risk in any dataset your team touches — spreadsheets, CRM exports, manually entered forms.

### 3.4 Deciding what to do about it

There's no single "correct" answer here — only trade-offs. A few options, and who might prefer which:

| Approach | What it does | Who might prefer it, and why |
|---|---|---|
| **Drop the rows** | Remove any listing with `price < €10,000` or `year_built < 1200` | A developer building a pipeline — simple, defensible, reproducible |
| **Cap / clip the values** | Set implausible values to a min/max threshold instead of removing them | An analyst who wants to keep the row's other data (area, location) intact |
| **Flag, don't touch** | Leave the data as-is but add an `is_suspicious` column | A designer building a dashboard — lets the *end user* see and decide, rather than hiding the decision |
| **Investigate further** | Go back to the source (Funda listing URL) to check if it's real | The most rigorous option — rarely done in practice due to time |

**For this workshop, we'll drop the clearest errors** so our later charts aren't distorted by a handful of bad rows — but keep in mind this is a *choice*, not a neutral default.


In [ ]:

before = len(df)

df_clean = df[
    (df["price"] > 10_000) &
    (df["year_built"] > 1200) &
    (df["year_built"] <= 2016)  # dataset appears to run through 2016
].copy()

after = len(df_clean)
print(f"Removed {before - after} rows ({(before - after) / before:.1%} of the dataset)")
print(f"Remaining: {after:,} rows")



> 🎯 **Optional / Bonus:** try changing the thresholds above (e.g. `price > 50_000`, or `year_built > 1500` to keep the real canal houses) and re-run. Watch how many rows get removed each time — small threshold decisions can meaningfully shift your dataset size and, later, your conclusions.

### 3.5 One more feature worth creating: price per m²

Raw price alone conflates "expensive" with "big." A more useful comparison is **price per square meter** — the actual metric used in real estate analysis.


In [ ]:

df_clean["price_per_m2"] = df_clean["price"] / df_clean["area"]
df_clean[["address", "area", "price", "price_per_m2"]].sort_values("price_per_m2", ascending=False).head()



## 4. What actually correlates with price? 📊

Now that the data is in reasonable shape, let's look at which features move together with `price`.

**Before running the cell — discuss for a moment:** what do you *expect* to correlate strongly with price? Area? Number of rooms? Age of the building?


In [ ]:

numeric_cols = ["area", "bedrooms", "rooms", "price", "year_built", "price_per_m2"]
corr = df_clean[numeric_cols].corr()

plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()



A few things worth pointing out once you see the heatmap:

- `area` and `price` are usually the strongest relationship — bigger places cost more, unsurprisingly.
- `bedrooms` and `rooms` correlate strongly *with each other* (as expected — they're not independent features), and both correlate less strongly with price than `area` does. This is worth sitting with: two features can each seem "important" individually while mostly just measuring the same underlying thing (size).
- `year_built`'s correlation with price is usually much weaker than people expect. Amsterdam's housing market doesn't reward "newer" the way people might assume — a beautifully located 1900s canal house can outprice a modern build.

> ⚠️ **The one line every data-adjacent person should say out loud at least once a year:**
> **Correlation is not causation.** A strong correlation between `area` and `price` doesn't prove that adding square meters *causes* a price increase by some fixed amount — location, property type, and market timing are tangled in there too.

### 4.1 Visualizing the strongest relationship


In [ ]:

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df_clean, x="area", y="price",
    hue="property_type", alpha=0.4, s=25
)
plt.title("Price vs. Area, by property type")
plt.xlabel("Area (m²)")
plt.ylabel("Price (€)")
plt.tight_layout()
plt.show()



> 🎯 **Optional / Bonus:** try re-coloring the scatterplot by a different column, e.g. `bedrooms` or the first 4 digits of `postal_code` (you'll need to extract it first: `df_clean["postal_code"].str[:4]`). Does a location pattern emerge?



## 5. 🎭 Two honest dashboards, two different stories

Here's the core exercise. Using the **exact same cleaned dataset**, we're going to build two small "dashboards" (a couple of charts each) — one that tells an **optimistic, reassuring** story about the Amsterdam housing market, and one that tells a **cautious, concerning** one.

**The rule: every chart must be technically accurate.** No fabricated numbers, no mislabeled axes. Just different, equally legitimate choices about:
- What time period or subset to show
- How to bin or group the data
- Which metric to lead with (mean vs. median, total vs. per m²)
- What to zoom in on vs. leave out

### 5.1 Dashboard A: "The market is healthy and accessible" 🟢

*(This uses `posting_date`, so let's first parse it into an actual date.)*


In [ ]:

df_clean["posting_date_parsed"] = pd.to_datetime(df_clean["posting_date"], format="%d-%m-%Y")
df_clean["posting_year"] = df_clean["posting_date_parsed"].dt.year

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: median price by year (medians resist the outlier-driven story)
median_by_year = df_clean.groupby("posting_year")["price"].median()
median_by_year.plot(kind="bar", ax=axes[0], color="#2a9d8f")
axes[0].set_title("Median asking price by year\n(a stable, modest metric)")
axes[0].set_ylabel("Median price (€)")
axes[0].set_xlabel("Year")

# Chart 2: apartments under 300k as a share of the market
affordable_share = (df_clean[df_clean["property_type"] == "apartment"]
                     .assign(is_affordable=lambda d: d["price"] < 300_000)
                     .groupby("posting_year")["is_affordable"].mean() * 100)
affordable_share.plot(kind="line", marker="o", ax=axes[1], color="#2a9d8f")
axes[1].set_title("Share of apartments listed under €300k")
axes[1].set_ylabel("% of apartment listings")
axes[1].set_xlabel("Year")
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()



### 5.2 Dashboard B: "The market is overheating and squeezing buyers out" 🔴

Same underlying data — different lens.


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: price per m2 by year (a less forgiving metric than raw price)
ppm2_by_year = df_clean.groupby("posting_year")["price_per_m2"].mean()
ppm2_by_year.plot(kind="bar", ax=axes[0], color="#e76f51")
axes[0].set_title("Average price per m² by year\n(a sharper, less forgiving metric)")
axes[0].set_ylabel("€ per m²")
axes[0].set_xlabel("Year")

# Chart 2: share of listings over 400k
expensive_share = (df_clean
                    .assign(is_expensive=lambda d: d["price"] > 400_000)
                    .groupby("posting_year")["is_expensive"].mean() * 100)
expensive_share.plot(kind="line", marker="o", ax=axes[1], color="#e76f51")
axes[1].set_title("Share of ALL listings priced over €400k")
axes[1].set_ylabel("% of listings")
axes[1].set_xlabel("Year")
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()



### 💬 Discuss as a group

- Both dashboards are built from **the same cleaned dataset**, with **no invented numbers**. Yet they leave very different impressions.
- Which one would a real estate agency want to show buyers? Which one would a tenants' rights group want to show?
- Look back at the specific choices that created the difference: median vs. mean, raw price vs. price-per-m², "under €300k" vs. "over €400k", apartments-only vs. all property types. Which of these choices would *you* have made without thinking twice?
- **For the designers in the room:** which chart *design* choices (color, framing, order) reinforce each dashboard's story on top of the underlying number choices?

> 🎯 **Optional / Bonus:** build a third dashboard — the *most neutral, least persuasive* version you can make of the same data. Is it actually possible to be neutral, or does every choice still lean somewhere?



## 6. 🎁 Optional / Bonus: predicting price

This section is entirely optional — skip it and stay in discussion if you'd rather. For anyone curious what a *very* simple version of "data science" (as opposed to data analysis) looks like, here's a minimal model that predicts price from a few features.

**Important framing:** this is a toy example to illustrate the idea, not a production-quality model. We are deliberately **not** doing the deeper work (train/test splits done properly, feature engineering, handling categorical variables carefully, model validation) a real project would need — that's a whole separate workshop.


In [ ]:

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

features = ["area", "bedrooms", "rooms", "year_built"]
X = df_clean[features]
y = df_clean["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Model coefficients (how much each feature moves the predicted price):")
for feature, coef in zip(features, model.coef_):
    print(f"  {feature:12s}: €{coef:,.0f} per unit")

print(f"\nMean Absolute Error: €{mean_absolute_error(y_test, predictions):,.0f}")
print(f"R² score: {r2_score(y_test, predictions):.2f}  (1.0 = perfect, 0.0 = no better than guessing the average)")



### 6.1 How wrong is the model, visually?


In [ ]:

plt.figure(figsize=(7, 6))
plt.scatter(y_test, predictions, alpha=0.4, s=25)
lims = [0, max(y_test.max(), predictions.max())]
plt.plot(lims, lims, color="black", linestyle="--", label="Perfect prediction")
plt.xlabel("Actual price (€)")
plt.ylabel("Predicted price (€)")
plt.title("Predicted vs. Actual price")
plt.legend()
plt.tight_layout()
plt.show()



**Discussion:** notice how far some points sit from the dashed "perfect prediction" line — those are listings this simple model badly misjudges. A real data scientist's job (closer to Anni's actual work) is largely about narrowing that spread: better features, better models, better validation. Today's version is intentionally the "quick sketch," not the finished building.

> 🎯 **Optional / Bonus:** add `property_type` as a feature (you'll need to convert it to a number first, e.g. `pd.get_dummies(df_clean["property_type"])`) and see if the R² score improves.



## 7. Wrap-up 💬

A few things worth carrying back to your own projects:

- **`.describe()` and `.info()` give you suspects, not verdicts.** Real investigation means looking at the actual rows behind a suspicious number.
- **Every cleaning decision is a design decision.** Dropping rows, capping values, or flagging them each shape the "truth" a viewer eventually sees — long before any chart is built.
- **The same honest data can tell very different stories** depending on metric choice, grouping, and framing. This isn't lying — it's just how data works, and it's exactly why the *presentation* choices your team makes carry real responsibility.
- **A data analyst and a data scientist do genuinely different jobs** — today lived mostly in analyst territory, with one small taste of the scientist's world at the end.

### Discussion to close
- Where has something like today's "price = €1,000" moment happened on a real client project?
- What would a "trust checklist" look like before a dashboard goes in front of a client — and who on the team should own which part of it?

Thanks for exploring some genuinely messy data with us. 🏠📊
